In [20]:
import os
os.remove(r"C:\Users\Raiyan\.config\earthengine\credentials")
print("Deleted")

Deleted


In [1]:
import json
with open(r"C:\Users\Raiyan\.config\earthengine\credentials") as f:
    creds = json.load(f)
print("Scopes:", creds.get('scopes'))

Scopes: ['https://www.googleapis.com/auth/earthengine', 'https://www.googleapis.com/auth/cloud-platform', 'https://www.googleapis.com/auth/drive', 'https://www.googleapis.com/auth/devstorage.full_control']


In [21]:
import json
import os

# Read the current (wrong) credentials first to get your tokens
# We'll just overwrite the scopes field

creds_path = r"C:\Users\Raiyan\.config\earthengine\credentials"

# First run ee.Authenticate() to get fresh tokens
import ee
ee.Authenticate(force=True)

# Now immediately overwrite the scopes with full permissions
with open(creds_path, 'r') as f:
    creds = json.load(f)

# Replace readonly scopes with full scopes
creds['scopes'] = [
    'https://www.googleapis.com/auth/earthengine',
    'https://www.googleapis.com/auth/drive',
    'https://www.googleapis.com/auth/cloud-platform'
]

with open(creds_path, 'w') as f:
    json.dump(creds, f)

print("Scopes updated!")
print("New scopes:", creds['scopes'])

Enter verification code:  4/1AfrIepAHRh09t3OIWSRxkHCiAPffzacs9ljbVI88MIctrPneRZjb_oqo9qE



Successfully saved authorization token.
Scopes updated!
New scopes: ['https://www.googleapis.com/auth/earthengine', 'https://www.googleapis.com/auth/drive', 'https://www.googleapis.com/auth/cloud-platform']


In [1]:
import ee
ee.Initialize(project='poverty-prediction-489716')
print("Initialized!")

Initialized!


In [2]:
test_fc = ee.FeatureCollection([
    ee.Feature(ee.Geometry.Point([90.4, 23.7]), {'test': 1})
])
test_task = ee.batch.Export.table.toDrive(
    collection=test_fc,
    description='permission_test',
    folder='GEE_exports',
    fileFormat='CSV'
)
test_task.start()
print(f"Status: {test_task.status()['state']}")

Status: READY


In [4]:
districts_fc = ee.FeatureCollection("FAO/GAUL/2015/level2") \
    .filter(ee.Filter.eq('ADM0_NAME', 'Bangladesh'))

print(f"Districts loaded: {districts_fc.size().getInfo()}")


Districts loaded: 64


In [5]:
viirs = ee.ImageCollection("NOAA/VIIRS/DNB/MONTHLY_V1/VCMCFG")

def extract_ntl_for_year(year):
    ntl_annual = viirs \
        .filter(ee.Filter.calendarRange(year, year, 'year')) \
        .select('avg_rad') \
        .median()
    
    stats = ntl_annual.reduceRegions(
        collection=districts_fc,
        reducer=ee.Reducer.mean()
            .combine(ee.Reducer.stdDev(),  sharedInputs=True)
            .combine(ee.Reducer.max(),     sharedInputs=True)
            .combine(ee.Reducer.min(),     sharedInputs=True)
            .combine(ee.Reducer.percentile([25, 75]), sharedInputs=True),
        scale=500
    )
    return stats.map(lambda f: f.set('year', year))

In [6]:
all_years = ee.FeatureCollection([])
for year in range(2018, 2023):
    yearly = extract_ntl_for_year(year)
    all_years = all_years.merge(yearly)
    print(f"Added year {year}")

print("All years merged!")

Added year 2018
Added year 2019
Added year 2020
Added year 2021
Added year 2022
All years merged!


In [7]:
task = ee.batch.Export.table.toDrive(
    collection=all_years,
    description='bangladesh_ntl_2018_2022',
    fileFormat='CSV',
    folder='GEE_exports',
    selectors=[
        'ADM2_NAME', 'ADM1_NAME', 'year',
        'mean', 'stdDev', 'max', 'min', 'p25', 'p75'
    ]
)
task.start()
print(f"Export started! Task ID: {task.id}")
print("Track progress: https://code.earthengine.google.com/tasks")

Export started! Task ID: JI6IWWZQLP4IDFDDPR6VYITA
Track progress: https://code.earthengine.google.com/tasks


In [9]:
status = task.status()
print(f"State:   {status['state']}")
print(f"Message: {status.get('error_message', 'none')}")

State:   COMPLETED
Message: none


In [10]:
worldpop = ee.ImageCollection("WorldPop/GP/100m/pop") \
    .filter(ee.Filter.eq('country', 'BGD')) \
    .filter(ee.Filter.eq('year', 2020)) \
    .first()

pop_stats = worldpop.reduceRegions(
    collection=districts_fc,
    reducer=ee.Reducer.mean().combine(
        ee.Reducer.sum(), sharedInputs=True),
    scale=100
)

task_pop = ee.batch.Export.table.toDrive(
    collection=pop_stats,
    description='bangladesh_population_2020',
    fileFormat='CSV',
    folder='GEE_exports',
    selectors=['ADM2_NAME', 'ADM1_NAME', 'mean', 'sum']
)
task_pop.start()
print(f"WorldPop export started: {task_pop.status()['state']}")

WorldPop export started: READY


In [11]:
modis_ndvi = ee.ImageCollection("MODIS/061/MOD13A3") \
    .filter(ee.Filter.date('2022-01-01', '2022-12-31')) \
    .select('_1_km_monthly_NDVI') \
    .mean() \
    .multiply(0.0001)

ndvi_stats = modis_ndvi.reduceRegions(
    collection=districts_fc,
    reducer=ee.Reducer.mean().combine(
        ee.Reducer.stdDev(), sharedInputs=True),
    scale=1000
)

task_ndvi = ee.batch.Export.table.toDrive(
    collection=ndvi_stats,
    description='bangladesh_ndvi_2022',
    fileFormat='CSV',
    folder='GEE_exports',
    selectors=['ADM2_NAME', 'ADM1_NAME', 'mean', 'stdDev']
)
task_ndvi.start()
print(f"NDVI export started: {task_ndvi.status()['state']}")

NDVI export started: READY


In [12]:
worldcover = ee.ImageCollection("ESA/WorldCover/v200").first()

# Class 50 = urban, mean of binary mask = fraction of urban pixels
urban_mask = worldcover.eq(50).rename('urban')
water_mask = worldcover.eq(80).rename('water')

lc_stats = urban_mask.addBands(water_mask).reduceRegions(
    collection=districts_fc,
    reducer=ee.Reducer.mean(),
    scale=100
)

task_lc = ee.batch.Export.table.toDrive(
    collection=lc_stats,
    description='bangladesh_landcover_2021',
    fileFormat='CSV',
    folder='GEE_exports',
    selectors=['ADM2_NAME', 'ADM1_NAME', 'urban', 'water']
)
task_lc.start()
print(f"Land cover export started: {task_lc.status()['state']}")

Land cover export started: READY


In [13]:
srtm = ee.Image('USGS/SRTMGL1_003')

elev_stats = srtm.reduceRegions(
    collection=districts_fc,
    reducer=ee.Reducer.mean().combine(
        ee.Reducer.stdDev(), sharedInputs=True),
    scale=90
)

task_elev = ee.batch.Export.table.toDrive(
    collection=elev_stats,
    description='bangladesh_elevation',
    fileFormat='CSV',
    folder='GEE_exports',
    selectors=['ADM2_NAME', 'ADM1_NAME', 'mean', 'stdDev']
)
task_elev.start()
print(f"Elevation export started: {task_elev.status()['state']}")

Elevation export started: READY


In [21]:
tasks = {
    'WorldPop':   task_pop,
    'NDVI':       task_ndvi,
    'LandCover':  task_lc,
    'Elevation':  task_elev,
}

for name, task in tasks.items():
    status = task.status()
    print(f"{name:12} → {status['state']}")

WorldPop     → COMPLETED
NDVI         → FAILED
LandCover    → COMPLETED
Elevation    → COMPLETED


In [22]:
# Check available MODIS bands first
modis_check = ee.ImageCollection("MODIS/061/MOD13A3").first()
print(modis_check.bandNames().getInfo())

['NDVI', 'EVI', 'DetailedQA', 'sur_refl_b01', 'sur_refl_b02', 'sur_refl_b03', 'sur_refl_b07', 'ViewZenith', 'SolarZenith', 'RelativeAzimuth', 'SummaryQA']


In [24]:
# Fixed — correct band name
modis_ndvi = ee.ImageCollection("MODIS/061/MOD13A3") \
    .filter(ee.Filter.date('2022-01-01', '2022-12-31')) \
    .select('NDVI') \
    .mean() \
    .multiply(0.0001)

ndvi_stats = modis_ndvi.reduceRegions(
    collection=districts_fc,
    reducer=ee.Reducer.mean().combine(
        ee.Reducer.stdDev(), sharedInputs=True),
    scale=1000
)

task_ndvi2 = ee.batch.Export.table.toDrive(
    collection=ndvi_stats,
    description='bangladesh_ndvi_2022',
    fileFormat='CSV',
    folder='GEE_exports',
    selectors=['ADM2_NAME', 'ADM1_NAME', 'mean', 'stdDev']
)
task_ndvi2.start()
print(f"NDVI retry started: {task_ndvi2.status()['state']}")

NDVI retry started: READY


In [28]:
print(f"NDVI status: {task_ndvi2.status()['state']}")

NDVI status: COMPLETED
